# 01 — Build the masked CD4 GENIE3 graph (regulatory prior for GREmLN)

Constructs the **raw masked CD4 GENIE3** co-expression graph used both as the GENIE3 comparator and
as GREmLN's regulatory prior (notebook 02). Steps: CD4 **non-targeting-control (NTC)** cell
selection → donor/batch audit → CPM/log1p normalisation → **ComBat** donor masking → TF regulator
universe → **GENIE3** (tree ensembles) → top-50,000-edge graph → model registry, plus a **SCENIC
top-50k sensitivity** (motif support only; **not** the GREmLN prior).

The GENIE3 forest fit is hours-long, so it is behind `REGENERATE`. By default the notebook
**reuses** the persisted `grn_edges.tsv` and audits it. Parameters come from
`config/benchmark_config.yaml`; see `data/README.md` to obtain the CD4 Perturb-seq NTC cells.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "scripts" / "bench_utils.py").exists()), _here.parent)
sys.path.insert(0, str(_root / "scripts"))
import bench_utils as bu
import yaml

CFG = yaml.safe_load((bu.repo_root() / "config" / "benchmark_config.yaml").read_text())
REGENERATE = False   # True re-runs GENIE3 (hours; needs the CD4 NTC expression matrix)
REG = bu.repo_root() / "results" / "model_registry"; REG.mkdir(parents=True, exist_ok=True)
print("GENIE3 params:", CFG["genie3"]); print("DATA_ROOT:", bu.data_root())

## 1. Source data, NTC selection, donor/batch audit, normalisation, ComBat masking

The GENIE3 input is CD4 **NTC** cells from the CZI genome-scale T-cell Perturb-seq resource:
select NTC guides, keep CD4, take the top-4,000 highly variable genes, normalise to CPM(1e6)→log1p,
then apply **ComBat on `donor_id`** (this donor-masking defines the *masked* graph). The exact
preprocessing lives in the development archive's prep script; parameters are pinned in the config.
The GREmLN prior in notebook 02 deliberately uses the *pre-ComBat* log1p-CPM of the same cells.

In [ ]:
# Documented preprocessing (guarded). In reuse mode we do not need the raw matrix.
if REGENERATE:
    import anndata as ad, scanpy as sc
    adata = ad.read_h5ad(bu.artifact("cd4_ntc_counts"))          # provide via BTLA_BENCH_CD4_NTC_COUNTS
    adata = adata[adata.obs["guide_target"].astype(str).eq("non-targeting")].copy()
    sc.pp.highly_variable_genes(adata, n_top_genes=CFG["genie3"]["n_hvg"], flavor="seurat_v3")
    adata = adata[:, adata.var["highly_variable"]].copy()
    sc.pp.normalize_total(adata, target_sum=1e6); sc.pp.log1p(adata)
    sc.pp.combat(adata, key="donor_id")                          # donor masking -> masked graph
    donor_audit = adata.obs["donor_id"].value_counts()
    print("cells x genes:", adata.shape); print(donor_audit)
else:
    print("REGENERATE=False -> reuse persisted masked graph; preprocessing shown above for provenance.")

## 2. TF regulator universe + GENIE3 (tree-ensemble feature importances)

GENIE3 fits, for each target gene, a tree ensemble predicting it from the TF regulators; edge weight
= regulator feature importance. Core algorithm inlined below; the full VIM over ~4k genes is the
hours-long step.

In [ ]:
def genie3_single(expr, target_idx, reg_idx, ntrees=1000, K="sqrt"):
    # Predict ONE target gene from the regulator set; return per-regulator importances.
    from sklearn.ensemble import ExtraTreesRegressor
    y = np.nan_to_num(np.asarray(expr[:, target_idx], float)); s = y.std()
    if s > 0: y = y / s
    preds = [i for i in reg_idx if i != target_idx]   # regulators are the predictors
    Xr = np.nan_to_num(np.asarray(expr[:, preds], float))
    est = ExtraTreesRegressor(n_estimators=ntrees, max_features=K, n_jobs=1).fit(Xr, y)
    imp = np.array([e.tree_.compute_feature_importances(normalize=False) for e in est.estimators_])
    vi = np.zeros(expr.shape[1]); vi[preds] = imp.sum(0) / len(est); return vi

def run_genie3(expr, gene_names, regulators, n_edges_keep=50000):
    # Standard GENIE3: every gene is a candidate TARGET predicted from the regulators;
    # edge = regulator -> target, weight = regulator's feature importance for that target.
    ng = expr.shape[1]; ridx = [i for i, g in enumerate(gene_names) if g in regulators]
    VIM = np.zeros((ng, ng))                          # VIM[target, regulator]
    for t in range(ng):                              # loop over ALL genes as targets
        VIM[t, :] = genie3_single(expr, t, ridx)
    links = [(gene_names[r], gene_names[t], float(VIM[t, r]))   # regulator r -> target t
             for t in range(ng) for r in ridx if r != t and VIM[t, r] > 0]
    links.sort(key=lambda x: x[2], reverse=True)
    return pd.DataFrame(links[:n_edges_keep], columns=["regulator", "target", "weight"])

if REGENERATE:
    tfs = bu.load_tfs(); gnames = list(adata.var_names.astype(str))
    regs = [g for g in gnames if g in tfs]
    edges = run_genie3(np.asarray(adata.X, float), gnames, regs, CFG["genie3"]["n_edges_persisted"])
    edges.to_csv(bu.artifact("genie3_edges"), sep="\t", index=False)
else:
    print("Skipping GENIE3 fit (reuse mode).")

## 3. Load / audit the persisted masked graph + model registry

In [ ]:
edges = bu.load_edges()
regs, tgts = set(edges["regulator"]), set(edges["target"])
tfs = bu.load_tfs()
stats = {"n_edges": len(edges), "n_regulators": len(regs), "n_regulators_in_tf_list": len(regs & tfs),
         "n_targets": len(tgts), "n_nodes": len(regs | tgts),
         "weight_min": float(edges["weight"].min()), "weight_max": float(edges["weight"].max())}
print(json.dumps(stats, indent=2))
registry = pd.DataFrame([
    ("model_name", "GENIE3_CD4_masked_raw"),
    ("graph_path", bu.redact(bu.artifact("genie3_edges"))),
    ("graph_md5", bu.md5(bu.artifact("genie3_edges"))),
    ("n_edges", stats["n_edges"]), ("n_regulators", stats["n_regulators"]),
    ("n_targets", stats["n_targets"]),
    ("n_hvg", CFG["genie3"]["n_hvg"]), ("normalisation", CFG["genie3"]["normalisation"]),
    ("batch_correction", CFG["genie3"]["batch_correction"]),
    ("tf_list", CFG["genie3"]["tf_list"]),
    ("tf_list_md5", bu.md5(bu.repo_root() / "resources" / "human_tfs_pySCENIC.txt")),
    ("tree_method", "ExtraTrees/RandomForest, ntrees=1000, K=sqrt"),
], columns=["field", "value"])
registry.to_csv(REG / "genie3_model_registry.csv", index=False)
registry

## 4. SCENIC top-50k sensitivity (motif support only — NOT the GREmLN prior)

> SCENIC pruning of the persisted top-50k GENIE3 graph is retained as an exploratory motif-support
> sensitivity analysis and is **not** used as the GREmLN prior. Enable with `REGENERATE` + pySCENIC
> databases; the pruned graph is reported separately and never fed to GREmLN.

In [ ]:
if REGENERATE:
    print("Run pySCENIC prune_targets on grn_edges.tsv with cisTarget motif rankings (see archive).")
else:
    print("SCENIC sensitivity skipped; documented as a separate motif-support analysis only.")